# 07b · Compositional dynamics: do tasks reuse dynamical building blocks?

**Investigation ladder** (see [`EVALUATION.md`](../EVALUATION.md)):
- **Stage 5 · Dynamics: is movement input-driven, and what structure helps a decoder?**

**Read first:** Lazzari & Saxena 2025, *Multitasking recurrent networks utilize compositional strategies for control of movement* (bioRxiv 10.1101/2025.09.10.675375); Linderman et al. 2017 (recurrent switching linear dynamical systems); the companion notebook [`07a_dynamics.ipynb`](07a_dynamics.ipynb) (single linear and physical models)

**We expect:** a few linear dynamical **motifs** are shared by many tasks (used in different mixtures), and they predict held-out hand movement better than one linear model. A multitask RNN with a task input decodes pose better than each recording's typical posture, and a new task can be learned from a few seconds by fitting **only its task-input vector**, faster than retraining the whole network. **Wrong if:** the motifs don't beat a single linear model on held-out data, every task uses its own private motif, the RNN ties with the per-recording median, or fitting the task vector is no better than no task input.

**The idea from the paper, and what is different here.** Lazzari & Saxena trained one recurrent network to make ten
movements through a **simulated** 6-muscle arm, with a task ("rule") input telling it which movement to make. The
network reused pieces: tasks shared a low-dimensional activity space and **dynamical motifs**, and a held-out task could
be learned by fitting **only its rule-input vector** while everything else stayed frozen. That is what they call a
compositional strategy.

We can't train a network to *move* a real hand. Instead we fit models to **real EMG and pose from one person** and ask
whether the same signatures appear in a model of our data:
- **Part 1 · Switching motifs:** do a few linear dynamical building blocks, shared across the 15 tasks, describe and
  predict how the hand moves?
- **Part 2 · A multitask RNN:** EMG and a task input → pose, trained with the paper's regularisers. Does it decode, do
  tasks share its activity space, and can a new task be learned by fitting only a task vector? That last test is the
  within-person version of the proposal's hypothesis H1 (less calibration for something new).

Results from a model fitted to data say what *a* model can do with this data, not what the nervous system does.

**How this notebook works** (the same in every notebook)
- **`# TODO`** cells are yours. Each has a docstring with the input and output shapes. You write the core
  algorithms yourself in NumPy.
- Under every TODO there are three **💡 hints** that open one at a time: the plan in words, the tools with a tiny
  example, then a skeleton with blanks (`____`).
- After each TODO a **✅ check cell** compares your answer with the reference in [`ref07b.py`](ref07b.py), then a **📊 plot**
  shows what your code did. Other notebooks use the reference, so your blanks never break theirs.
- **✍️ questions** are for your own words, in the markdown cell under them.
- The **Result** cell at the end is the **gate**: it decides whether the next stage of the investigation is worth running.

## 🎯 Goal and milestones
*Draft milestones: agree them as a team on Friday, then edit here.*

**Proposal:** ➕ beyond the proposal, but it is the compositional modelling the team asked for, and its transfer test is a within-person pilot of H1 (stage 6/7 ask the same question across bottlenecks and people).

**Goal:** test whether shared, reusable dynamical building blocks describe and predict the hand, and whether a new task can be learned by recombining what a network already knows.

**Why:** if movement is built from reusable pieces, a decoder that knows the pieces should need little data for something new. That is the logic behind H1.

**Milestones**
1. Switching-motif model written from scratch; K chosen by held-out error
2. Motif usage per task, reuse count and mutual information against a shuffled null
3. A held-out task: described by the other tasks' motifs vs its own
4. A regularised multitask RNN; held-out error against the per-recording median and a shifted-EMG null, with CIs over blocks
5. Calibration curve on a held-out task: task vector only vs all weights vs no task vs nearest task
6. Shared subspaces of the RNN's activity vs its EMG input and pose output

**Gate (decides the next stage):** (a) motifs predict held-out movement better than one linear model; (b) the RNN beats the per-recording median **and** its shifted-EMG input (paired CIs over blocks exclude 0); (c) fitting only the task vector beats no task input with ≤ 5 s of a new task.

**Done when:** every ✅ check passes and the gate table is filled in.

In [ ]:
# ── Setup: works on Colab and locally ─────────────────────────────────────────
# After this cell the working directory is the repo root, wherever you run from.
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ or Path("/content").exists()
if IN_COLAB:
    ROOT = Path("/content/emg2pose")
    if not ROOT.exists():
        !git clone https://github.com/Wilbertbh-Tan/emg2pose.git {ROOT}
    else:
        !git -C {ROOT} pull -q                 # pick up files pushed since the first clone
    !pip install -q h5py
else:
    ROOT = Path.cwd().resolve()
    while not (ROOT / "setup.py").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    assert (ROOT / "setup.py").exists(), "Run this notebook from inside the emg2pose repo"
os.chdir(ROOT)

_d = ROOT / "notebooks" / "experiments" / "07_dynamics"
assert (_d / "ref07b.py").exists(), f"{_d}/ref07b.py is missing: pull the latest version of the repo"
sys.path.insert(0, str(_d))
sys.modules.pop("ref07b", None)
_d = ROOT / "notebooks" / "experiments" / "06_shared_subspace"
assert (_d / "ref06.py").exists(), f"{_d}/ref06.py is missing: pull the latest version of the repo"
sys.path.insert(0, str(_d))
sys.modules.pop("ref06", None)
_d = ROOT / "notebooks" / "experiments" / "02_how_many_synergies"
assert (_d / "ref02.py").exists(), f"{_d}/ref02.py is missing: pull the latest version of the repo"
sys.path.insert(0, str(_d))
sys.modules.pop("ref02", None)
_d = ROOT / "notebooks" / "experiments" / "common"
sys.path.insert(0, str(_d)); sys.modules.pop("protocol", None)
import protocol                 # shared evaluation protocol: sessions, folds, baselines, nulls, CIs
import ref07b as ref          # the reference answers, used only by the ✅ check cells

PROCESSED = ROOT / "data" / "processed_mini.npz"   # made by notebook 01's prep.py (data/ is never committed)

In [ ]:
# ── Load the shared data (rebuilds it with prep.py if missing or built by an older version) ──
import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0, str(ROOT / "notebooks" / "experiments" / "01_data_prep")); sys.modules.pop("prep", None)
import prep
if not prep.is_current(PROCESSED):
    subprocess.run([sys.executable, "notebooks/experiments/01_data_prep/prep.py"], check=True)
d = np.load(PROCESSED)
import torch
import ref06, ref02

emg        = d["emg"].astype(np.float64)    # (N, 16) EMG envelope, 50 frames/s
pose       = d["pose"].astype(np.float64)   # (N, 20) joint angles in degrees (low-passed at 6 Hz, data version 2)
pose_valid = d["pose_valid"]                # (N,)    False where tracking failed (dropouts padded by 40 ms)
rec        = d["rec"]                       # (N,)    recording of each frame (0–29)
split      = d["split"]                     # (N,)    0 train, 1 test, -1 gap
stage_of_rec, side_of_rec, filename_of_rec = d["stage"], d["side"], d["filename"]
train, test = split == 0, split == 1

block = protocol.block_of_rec(filename_of_rec)[rec]              # (N,) independent unit (15 blocks)
groups = protocol.stage_groups(stage_of_rec, filename_of_rec)    # 14 task groups (free style + idle hand merged)
group_names = [name for name, _ in groups]
group_of_rec = np.zeros(len(stage_of_rec), int)
for g, (_, recs) in enumerate(groups):
    group_of_rec[recs] = g
group = group_of_rec[rec]                                         # (N,) task group of every frame
G = len(groups)

SEED = 0
torch.manual_seed(SEED)
torch.set_num_threads(min(4, os.cpu_count() or 1))
print(f"{len(emg)} frames | {G} task groups | {len(np.unique(block))} blocks | torch {torch.__version__}")

### The variables you start with
| Name | Shape | What it is |
|---|---|---|
| `emg`, `pose` | (N, 16), (N, 20) | EMG envelope; joint angles in degrees (pose cleaned in notebook 01, Task 5b) |
| `pose_valid`, `rec`, `split`, `train`, `test` | (N,) | as in every notebook |
| `block` | (N,) | the independent unit: 15 simultaneous left/right recording periods |
| `groups`, `group_names`, `group`, `G` | 14 | **task groups**: each stage, except that `OneHandedFreeStyle` and the idle hand's `unconstrained` recordings were made at the same time, so they form one group |

Letters in docstrings: **S** segments, **L** steps per segment, **d** state dimensions, **m** input dimensions,
**K** motifs, **H** hidden units, **B** batch.

## Before you run anything: your prediction

Write what you expect and why (2–3 sentences). Compare at the end. Surprises are the interesting part.

> *Prediction:*

---
## Part 1 · Switching linear motifs
Notebook 07a fits **one** linear model to the whole hand: `z(t+1) = A z(t) + B u(t) + b`. But a hand opening and a
hand closing may follow different dynamics. A **switching** model keeps a small library of K linear models
("motifs") and lets each short piece of movement use one of them. Recurrent switching linear dynamical systems
(rSLDS; Linderman et al. 2017) do this with probabilities; we use the simplest version, **hard EM**:

1. Cut the data into **segments** of 10 steps (200 ms).
2. Give every segment a random motif label.
3. **M-step:** refit each motif by least squares on the segments that use it.
4. **E-step:** give each segment the motif with the smallest one-step error.
5. Repeat 3–4. Like k-means, the total error can only go down.

**State and input.** To keep each motif small we work in low dimensions:
- the state `z` = the top 4 PCA scores of the pose (notebook 02's eigenpostures);
- the input `u` = the 3 **shared EMG latents** from notebook 06 (the EMG directions that predict pose), taken 200 ms
  earlier, since EMG leads movement.

Each motif then has 4×4 + 4×3 + 4 = 32 numbers.

In [ ]:
# Given: state, input and segments
D_STATE, RANK, LAG = 4, 3, 10                     # 4 pose PCs; 3 shared EMG latents; EMG taken 10 frames (200 ms) earlier
tr_v, te_v = train & pose_valid, test & pose_valid
pmean, pcomps = ref02.pca_fit(pose[tr_v], D_STATE)
Z = ref02.pca_transform(pose, pmean, pcomps)                              # (N, 4)  state
X_lag = ref06.lag_features(ref06.log_emg(emg), rec, [LAG])               # (N, 16) log-EMG 200 ms earlier
B_r, x_mean, y_mean, _ = ref06.rrr_fit(X_lag[tr_v], pose[tr_v], RANK)
U_lat = ref06.shared_latents(X_lag, B_r, x_mean, RANK)                   # (N, 3)  input

s_tr, s_te = ref.make_segments(rec, tr_v), ref.make_segments(rec, te_v)  # segment start frames
tri_tr = ref.segment_triples(Z, U_lat, s_tr)                              # (z(t), u(t), z(t+1)), each (S, 10, ·)
tri_te = ref.segment_triples(Z, U_lat, s_te)
print(f"{len(s_tr)} training and {len(s_te)} test segments of 10 steps")

### Task 1 · Fit motifs with hard EM
Write four functions:
- `fit_motif`: least squares for one motif. Put the regressors side by side, `F = [z, u, 1]`, and solve the
  ridge normal equations `(FᵀF + αI) W = Fᵀ z_next`. The rows of `W` hold `Aᵀ`, `Bᵀ` and `b`.
- `step`: one step of a motif, `z Aᵀ + u Bᵀ + b`, working on arrays of any leading shape.
- `motif_errors`: for every segment and every motif, the summed squared one-step error.
- `fit_switching`: the hard-EM loop. If a motif ends up with fewer than 5 segments, restart it on the
  worst-fit segments (the reference uses the worst `max(5, S // (4K))`).

In [ ]:
def fit_motif(Zt: np.ndarray, Ut: np.ndarray, Zn: np.ndarray, alpha: float = 1e-3
              ) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Ridge least squares for z(t+1) = A z(t) + B u(t) + b.

    Parameters
    ----------
    Zt : (..., d) states z(t); Ut : (..., m) inputs u(t); Zn : (..., d) next states z(t+1) (any leading shape)
    alpha : small ridge penalty

    Returns
    -------
    A : (d, d), B : (d, m), b : (d,)
    """
    # TODO
    raise NotImplementedError


def step(z: np.ndarray, u: np.ndarray, motif: tuple[np.ndarray, np.ndarray, np.ndarray]) -> np.ndarray:
    """One step z(t+1) = A z + B u + b for (..., d) z and (..., m) u. Returns (..., d)."""
    # TODO
    raise NotImplementedError


def motif_errors(Zt: np.ndarray, Ut: np.ndarray, Zn: np.ndarray, motifs: list) -> np.ndarray:
    """(S, K): summed squared one-step error of every segment (over its L steps and d dims) under every motif.
    Zt, Ut, Zn have shapes (S, L, ·)."""
    # TODO
    raise NotImplementedError


def fit_switching(Zt: np.ndarray, Ut: np.ndarray, Zn: np.ndarray, K: int, n_iter: int = 20, seed: int = 0,
                  alpha: float = 1e-3) -> tuple[list, np.ndarray, list[float]]:
    """Hard-EM switching linear dynamics.

    Returns
    -------
    motifs  : list of K (A, B, b)
    labels  : (S,) int, the motif each segment uses after the last E-step
    history : list of n_iter floats, the total error (sum over segments of their best motif's error) after each E-step
    Start with labels = np.random.default_rng(seed).integers(0, K, size=S).
    """
    # TODO
    raise NotImplementedError

#### 💡 Stuck on Task 1? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. `fit_motif`: flatten the leading dimensions (`reshape(-1, d)`), build `F` with a column of ones, solve, then split
   the rows of the solution: the first d rows are `Aᵀ`, the next m are `Bᵀ`, the last is `b`.
2. `step`: one line with `@` and transposes.
3. `motif_errors`: for each motif, `((step(Zt, Ut, mo) - Zn) ** 2).sum(axis=(1, 2))`, then stack the K columns.
4. `fit_switching`: random labels → loop: fit each motif on `labels == k` (restart if too few) → errors → new labels
   = argmin over motifs → record the total of the minimum errors.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `np.concatenate([a, b, c], axis=1)` glues columns; `np.ones((n, 1))` is the intercept column.
- `np.linalg.solve(M, R)` solves `M W = R`. `errs.argmin(axis=1)` picks the best motif per row.

```python
F = np.array([[1., 2., 1.], [3., 4., 1.]])      # two rows of [z, u, 1]
W = np.linalg.solve(F.T @ F + 1e-3 * np.eye(3), F.T @ np.array([[5.], [11.]]))
W[:1].T, W[1:2].T, W[-1]                          # A, B, b for d = m = 1
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def fit_motif(Zt, Ut, Zn, alpha=1e-3):
    d, m = Zt.shape[-1], Ut.shape[-1]
    F = np.concatenate([Zt.reshape(-1, d), Ut.reshape(-1, m), np.ones((Zt[..., 0].size, 1))], axis=1)
    W = np.linalg.solve(F.T @ F + alpha * np.eye(F.shape[1]), F.T @ Zn.reshape(-1, ____))
    return W[:d].T, W[d:d + m].T, W[____]

def step(z, u, motif):
    A, B, b = motif
    return z @ A.T + u @ ____ + b

def motif_errors(Zt, Ut, Zn, motifs):
    return np.stack([((step(Zt, Ut, mo) - Zn) ** 2).sum(axis=(1, 2)) for mo in ____], axis=1)

def fit_switching(Zt, Ut, Zn, K, n_iter=20, seed=0, alpha=1e-3):
    rng = np.random.default_rng(seed)
    S = len(Zt)
    labels = rng.integers(0, K, size=S)
    fit_err, history = rng.random(S), []
    for _ in range(n_iter):
        motifs = []
        for k in range(K):
            sel = labels == k
            if sel.sum() < 5:
                worst = np.argsort(fit_err)[-max(5, S // (4 * K)):]
                sel = np.zeros(S, bool); sel[worst] = True
            motifs.append(fit_motif(Zt[sel], Ut[sel], Zn[sel], alpha))
        errs = motif_errors(Zt, Ut, Zn, motifs)
        labels = errs.argmin(axis=____)
        fit_err = errs.min(axis=1)
        history.append(float(fit_err.sum()))
    return motifs, labels, history
```

</details>

In [ ]:
# ✅ Check Task 1
rng_c = np.random.default_rng(1)
Zc, Uc, Nc = rng_c.normal(size=(50, 10, 3)), rng_c.normal(size=(50, 10, 2)), rng_c.normal(size=(50, 10, 3))
for mine, theirs in zip(fit_motif(Zc, Uc, Nc), ref.fit_motif(Zc, Uc, Nc)):
    assert np.allclose(mine, theirs, atol=1e-8), "fit_motif differs: check the column order [z, u, 1] and the split of W"
mo_c = ref.fit_motif(Zc, Uc, Nc)
assert np.allclose(step(Zc, Uc, mo_c), ref.step(Zc, Uc, mo_c)), "step differs"
assert np.allclose(motif_errors(Zc, Uc, Nc, [mo_c, mo_c]), ref.motif_errors(Zc, Uc, Nc, [mo_c, mo_c]))

# A system with two known motifs: half the segments rotate one way, half the other
A1 = np.array([[0.95, -0.2], [0.2, 0.95]]); A2 = A1.T
zs, us, ns, truth = [], [], [], []
for s in range(200):
    A = A1 if s % 2 == 0 else A2
    z = rng_c.normal(size=2); seg_z, seg_n = [], []
    for t in range(10):
        nz = A @ z + 0.01 * rng_c.normal(size=2); seg_z.append(z); seg_n.append(nz); z = nz
    zs.append(seg_z); ns.append(seg_n); us.append(np.zeros((10, 1))); truth.append(s % 2)
zs, us, ns, truth = np.array(zs), np.array(us), np.array(ns), np.array(truth)
mot, lab, hist = fit_switching(zs, us, ns, K=2, seed=0)
assert all(b <= a * (1 + 1e-9) for a, b in zip(hist, hist[1:])), "the total error went up: E and M steps should only lower it"
agree = max(np.mean(lab == truth), np.mean(lab != truth))
assert agree > 0.95, f"only {agree:.0%} of segments recovered their true motif"
print(f"✅ Task 1 works: hard EM recovers two known motifs ({agree:.0%} of segments labelled correctly)")

### 📊 Task 1b · How many motifs? Held-out error for K = 1 … 6
More motifs always fit the *training* data better, so K is chosen on **test** segments. There is a catch: a segment's
motif has to be chosen *somehow* on new data. We use the honest option, `ref.infer_and_score`: choose the motif from
the **first half** of each test segment, then score the **second half** (one-step error, and the error of a free
rollout to the segment's end). K = 1 is a single linear model, the baseline. **What to look for:** does any K > 1 beat
K = 1 on held-out data?

In [ ]:
Ks = [1, 2, 3, 4, 5, 6]
one_step, rollout, persistence = [], [], []
prev_tr, has_prev = ref.predecessor(s_tr)
for K in Ks:
    mot_K, lab_K, _ = fit_switching(*tri_tr, K=K, seed=SEED)
    o, r_, _ = ref.infer_and_score(*tri_te, mot_K)
    one_step.append(o); rollout.append(r_)
    persistence.append(np.mean(lab_K[has_prev] == lab_K[prev_tr][has_prev]))     # same motif as the previous segment?
no_change = np.mean(((tri_te[0][:, 5] - tri_te[2][:, -1]) ** 2).sum(axis=1))      # 'the pose stays put' over 5 steps

fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].plot(Ks, one_step, "o-"); ax[0].set_title("held-out one-step error"); ax[0].set_xlabel("K motifs")
ax[1].plot(Ks, rollout, "o-", label="motif rollout"); ax[1].axhline(no_change, ls="--", color="gray", label="no change")
ax[1].set_title("held-out 5-step rollout error"); ax[1].set_xlabel("K motifs"); ax[1].legend()
for a in ax: a.set_ylabel("squared error (PCA units²)")
plt.tight_layout(); plt.show()
K_BEST = Ks[int(np.argmin(one_step))]
print("one-step:", np.round(one_step, 1), "| rollout:", np.round(rollout, 0), f"| no change: {no_change:.0f}")
print("same motif as the previous segment:", np.round(persistence, 2), f"| K chosen on held-out one-step error: {K_BEST}")

✍️ **Q1.** Which K wins on held-out data, and how does a single linear model (K = 1) compare with
"no change"? The persistence numbers say how often a segment keeps its predecessor's motif. If motifs switch almost as
often as chance, what are they probably capturing: different *kinds* of movement, or noise and the direction of each
small movement? Why does that make them useless for prediction, even if they describe the training data well?

> *Your answer:*

---
### Task 2 · Are motifs shared across tasks, or private?
Lazzari & Saxena's key question: do tasks **reuse** the same building blocks? Even if the motifs don't help
prediction, the way tasks *use* them still says whether the dynamics are shared. We look at K = 4 (a library big
enough to show structure; Task 1b is about prediction, this is description):
- `usage_matrix`: for each task group, the share of its segments that use each motif (rows sum to 1).
- `groups_per_motif`: how many groups use each motif for at least 10% of their segments. Private motifs → 1.
- `mutual_information` between motif and group, in bits: 0 = every group uses the **same mix** of motifs;
  larger = groups use **different mixes**. Composition means both: shared motifs, task-specific mixtures.

The null shuffles the group labels of the segments: the mutual information you'd see by chance.

$$I = \sum_{g,k} p(g,k) \log_2 \frac{p(g,k)}{p(g)\,p(k)}$$

In [ ]:
def usage_matrix(labels: np.ndarray, group_of_seg: np.ndarray, n_groups: int, K: int) -> np.ndarray:
    """(n_groups, K): share of each group's segments that use each motif. Rows sum to 1 (0 for an empty group)."""
    # TODO
    raise NotImplementedError


def groups_per_motif(M: np.ndarray, threshold: float = 0.1) -> np.ndarray:
    """(K,): number of groups whose usage of each motif is at least ``threshold``."""
    # TODO
    raise NotImplementedError


def mutual_information(labels: np.ndarray, groups: np.ndarray) -> float:
    """Mutual information (bits) between two integer label arrays of equal length."""
    # TODO
    raise NotImplementedError

#### 💡 Stuck on Task 2? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. `usage_matrix`: for each group, take its segments' labels, count each motif (`np.bincount(..., minlength=K)`),
   divide by the number of segments.
2. `groups_per_motif`: compare the matrix with the threshold and count down the rows.
3. `mutual_information`: build the joint count table (groups × motifs), turn it into probabilities, get the two
   marginals, then sum p·log2(p / (p_g p_k)) over the non-zero cells.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `np.add.at(table, (rows, cols), 1)` adds 1 at every (row, col) pair, even repeated ones.
- `p.sum(1, keepdims=True) @ p.sum(0, keepdims=True)` is the table of products p(g)·p(k).

```python
t = np.zeros((2, 2)); np.add.at(t, (np.array([0, 0, 1]), np.array([1, 1, 0])), 1)
t                                   # [[0, 2], [1, 0]]
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def usage_matrix(labels, group_of_seg, n_groups, K):
    M = np.zeros((n_groups, K))
    for g in range(n_groups):
        sel = group_of_seg == g
        if sel.any():
            M[g] = np.bincount(labels[sel], minlength=____) / sel.sum()
    return M

def groups_per_motif(M, threshold=0.1):
    return (M >= threshold).sum(axis=____)

def mutual_information(labels, groups):
    joint = np.zeros((groups.max() + 1, labels.max() + 1))
    np.add.at(joint, (groups, labels), 1)
    p = joint / joint.sum()
    expected = p.sum(1, keepdims=True) @ p.sum(0, keepdims=True)
    nz = p > 0
    return float((p[nz] * np.log2(p[nz] / ____[nz])).sum())
```

</details>

In [ ]:
# ✅ Check Task 2
lab_c = np.array([0, 0, 1, 1, 2, 2, 0, 1]); grp_c = np.array([0, 0, 0, 1, 1, 1, 2, 2])
assert np.allclose(usage_matrix(lab_c, grp_c, 3, 3), ref.usage_matrix(lab_c, grp_c, 3, 3))
assert np.array_equal(groups_per_motif(ref.usage_matrix(lab_c, grp_c, 3, 3)), ref.groups_per_motif(ref.usage_matrix(lab_c, grp_c, 3, 3)))
assert abs(mutual_information(lab_c, grp_c) - ref.mutual_information(lab_c, grp_c)) < 1e-12
assert abs(mutual_information(np.array([0, 1, 0, 1]), np.array([0, 0, 1, 1]))) < 1e-12, "independent labels should give 0 bits"
print("✅ Task 2 functions work")

# Given: motif reuse in the real data (K = 4)
K_REUSE = 4
mot4, lab4, _ = fit_switching(*tri_tr, K=K_REUSE, seed=SEED)
gseg = group[s_tr]
M4 = usage_matrix(lab4, gseg, G, K_REUSE)
reuse = groups_per_motif(M4)
mi = mutual_information(lab4, gseg)
rng_n = np.random.default_rng(SEED)
mi_null = np.array([mutual_information(lab4, rng_n.permutation(gseg)) for _ in range(200)])
print(f"groups using each motif (≥ 10% of their segments): {reuse}  (out of {G})")
print(f"mutual information motif × task: {mi:.3f} bits | shuffled null {mi_null.mean():.3f} (95th pct {np.quantile(mi_null, 0.95):.3f})")

### 📊 Motif usage per task
Rows: task groups; columns: motifs; colour: share of the task's segments. **What to look for:** columns that are
coloured in many rows (shared motifs) vs a column lit in one row only (a private motif); and whether rows differ in
*mix*.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(M4, cmap="viridis", vmin=0, vmax=1, aspect="auto")
ax.set_yticks(range(G), group_names, fontsize=7); ax.set_xticks(range(K_REUSE), [f"motif {k}" for k in range(K_REUSE)])
plt.colorbar(im, ax=ax, label="share of the task's segments"); ax.set_title("Motif usage per task group")
plt.tight_layout(); plt.show()

### A new task: existing motifs or a new one?
Hold out one task group, fit K = 4 motifs on the other groups, and ask how well that library **describes** the held-out
task's test segments, compared with a library of the same size fitted on the held-out task's own **training** segments.
Both scores let each segment pick its best motif (`ref.describe_error`), so the comparison is fair. A ratio near 1
means the new task is made of existing building blocks.

In [ ]:
HELD_OUT = "FingerFreeform"            # a free-style task: plausibly a mix of the instructed movements
hg = group_names.index(HELD_OUT)
other_tr = s_tr[group[s_tr] != hg]
own_tr, own_te = s_tr[group[s_tr] == hg], s_te[group[s_te] == hg]
lib_other, _, _ = fit_switching(*ref.segment_triples(Z, U_lat, other_tr), K=K_REUSE, seed=SEED)
lib_own, _, _ = fit_switching(*ref.segment_triples(Z, U_lat, own_tr), K=K_REUSE, seed=SEED)
tri_own_te = ref.segment_triples(Z, U_lat, own_te)
e_other, e_own = ref.describe_error(*tri_own_te, lib_other), ref.describe_error(*tri_own_te, lib_own)
e_single = ref.describe_error(*tri_own_te, [fit_motif(*ref.segment_triples(Z, U_lat, other_tr))])
print(f"{HELD_OUT} test segments, mean one-step error: other tasks' 4 motifs {e_other:.1f} | its own 4 motifs {e_own:.1f} "
      f"| one motif from other tasks {e_single:.1f}  → own ÷ other = {e_own / e_other:.2f}")

✍️ **Q2.** Are the motifs shared across tasks, private, or both (shared motifs, different mixes)? Use
the reuse counts, the mutual information against its null, and the held-out task's ratio. Lazzari & Saxena found shared
motifs in a network trained on ten arm movements. Does your answer depend on the motifs being good *predictors* (Task
1b)? Why or why not?

> *Your answer:*

---
## Part 2 · A multitask RNN
Now the paper's model class: a recurrent network that receives the EMG and a **task input**, and outputs the pose.
Its hidden state `h` (64 units) carries memory, so it can integrate the EMG over time (notebook 05 showed that the
link from EMG to pose only appears with history).

$$h_{t+1} = h_t + \alpha\left(-h_t + W_h \tanh(h_t) + W_u u_t + W_\text{task}\, e_\text{task} + b\right), \qquad
r_t = \tanh(h_t), \qquad y_t = W_o r_t + b_o$$

- `u_t`: the 16 log-EMG channels, standardised (no lag needed: the network has memory).
- `e_task`: a one-hot vector for the task group, so `W_task e_task` is that task's **input vector** (a column of
  `W_task`).
- `y_t`: the top 6 pose PCA scores, standardised; turned back into degrees with `ref02.pca_inverse`.
- `α = 0.2`: a time constant of 5 frames (100 ms).

**Loss** (the paper's terms): masked mean squared error on valid-pose frames, plus small L1 penalties on the rates
(few active units) and the weights, plus a "simple dynamics" penalty: the size of the recurrence's Jacobian
`W_h diag(1 − r²)`, which discourages chaotic, hard-to-interpret dynamics.

**Held-out task.** `FingerFreeform` is left out of training entirely; Task 4 uses it to test transfer.

In [ ]:
def rnn_step(h: torch.Tensor, u: torch.Tensor, task: torch.Tensor, p: dict, alpha: float) -> torch.Tensor:
    """One update. h (B, H), u (B, n_in), task (B, G) -> new h (B, H). p holds W_h, W_u, W_task, b, W_o, b_o."""
    # TODO
    raise NotImplementedError


def rnn_run(U: torch.Tensor, T: torch.Tensor, p: dict, alpha: float) -> tuple[torch.Tensor, torch.Tensor]:
    """Run sequences from h = 0. U (B, L, n_in), T (B, L, G) -> outputs y (B, L, n_out), rates r (B, L, H).
    Use r = tanh(h) after each update, and y = r W_o^T + b_o."""
    # TODO
    raise NotImplementedError


def rnn_loss(Y_hat: torch.Tensor, Y: torch.Tensor, M: torch.Tensor, R: torch.Tensor, p: dict,
             lam_rate: float, lam_weight: float, lam_dyn: float) -> tuple[torch.Tensor, torch.Tensor]:
    """Regularised loss. M (B, L) bool marks frames that count.
    mse   = sum of squared errors over counted frames and outputs / (counted frames × n_out)
    total = mse + lam_rate · mean|r| + lam_weight · (mean|W_h| + mean|W_u| + mean|W_o|)
                + lam_dyn · mean over (B, L) of  Σ_j (Σ_i W_h[i, j]²)(1 − r_j²)²  / H
    Returns (total, mse)."""
    # TODO
    raise NotImplementedError

#### 💡 Stuck on Task 3? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. `rnn_step`: the update equation, with `@` and `.T`. `torch.tanh` for the nonlinearity.
2. `rnn_run`: start `h` at zeros of shape (B, H); loop over the L time steps, update `h`, keep `tanh(h)`; stack the
   rates along a new time axis; the outputs are one matrix product.
3. `rnn_loss`: turn the mask into weights, compute the masked MSE, then the three penalties exactly as in the docstring.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `torch.zeros(B, H)`; `torch.stack(list_of_(B,H), dim=1)` → (B, L, H).
- `M.unsqueeze(-1).float()` turns a (B, L) mask into (B, L, 1) weights that broadcast over outputs.
- `(p["W_h"] ** 2).sum(0)` is (H,): column j's squared norm.

```python
h = torch.zeros(2, 3); u = torch.ones(2, 1)
W = torch.ones(3, 1)
h + 0.2 * (-h + u @ W.T)        # every unit moves 20% of the way to 1
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def rnn_step(h, u, task, p, alpha):
    return h + alpha * (-h + torch.tanh(h) @ p["W_h"].T + u @ p["W_u"].T + task @ p["____"].T + p["b"])

def rnn_run(U, T, p, alpha):
    B, L, _ = U.shape
    h = torch.zeros(B, p["W_h"].shape[0])
    rates = []
    for t in range(L):
        h = rnn_step(h, U[:, t], T[:, t], p, alpha)
        rates.append(torch.tanh(____))
    R = torch.stack(rates, dim=1)
    return R @ p["W_o"].T + p["b_o"], R

def rnn_loss(Y_hat, Y, M, R, p, lam_rate, lam_weight, lam_dyn):
    w = M.unsqueeze(-1).float()
    mse = ((Y_hat - Y) ** 2 * w).sum() / (w.sum() * Y.shape[-1]).clamp(min=1)
    l_rate = R.abs().mean()
    l_w = p["W_h"].abs().mean() + p["W_u"].abs().mean() + p["W_o"].abs().mean()
    l_dyn = ((p["W_h"] ** 2).sum(0) * (1 - R ** 2) ** 2).sum(-1).mean() / p["W_h"].shape[0]
    return mse + lam_rate * l_rate + lam_weight * ____ + lam_dyn * l_dyn, mse
```

</details>

In [ ]:
# ✅ Check Task 3
p_c = ref.rnn_params(5, 3, 8, 2, seed=1)
U_c, T_c = torch.randn(2, 7, 5), torch.eye(3)[torch.tensor([[0] * 7, [2] * 7])]
h_c = torch.randn(2, 8)
assert torch.allclose(rnn_step(h_c, U_c[:, 0], T_c[:, 0], p_c, 0.2), ref.rnn_step(h_c, U_c[:, 0], T_c[:, 0], p_c, 0.2), atol=1e-6)
Yh_c, R_c = rnn_run(U_c, T_c, p_c, 0.2); rYh, rR = ref.rnn_run(U_c, T_c, p_c, 0.2)
assert Yh_c.shape == (2, 7, 2) and R_c.shape == (2, 7, 8), f"shapes {tuple(Yh_c.shape)}, {tuple(R_c.shape)}"
assert torch.allclose(Yh_c, rYh, atol=1e-6) and torch.allclose(R_c, rR, atol=1e-6), "rnn_run differs: start from h = 0, rates = tanh(h)"
Y_c, M_c = torch.randn(2, 7, 2), torch.rand(2, 7) > 0.3
regs_c = dict(lam_rate=0.1, lam_weight=0.1, lam_dyn=0.1)
for a, b in zip(rnn_loss(Yh_c, Y_c, M_c, R_c, p_c, **regs_c), ref.rnn_loss(Yh_c, Y_c, M_c, R_c, p_c, **regs_c)):
    assert torch.allclose(a, b, atol=1e-6), "rnn_loss differs: check the masked MSE and the three penalties"
print("✅ Task 3 works: your RNN matches the reference")

### Train it (given)
The training loop is given: Adam on random batches of 2-second chunks from the **training** part of every recording
except the held-out task, ignoring the first 10 frames of each chunk (the network starts from h = 0). It uses **your**
`rnn_run` and `rnn_loss`. About 20 s on a laptop CPU; sizes are kept small for speed (64 units, 12 passes).

In [ ]:
# Given: inputs, targets and the training loop
H, ALPHA, D_OUT = 64, 0.2, 6
EPOCHS, LR, CHUNK, STRIDE, BURN = 12, 5e-3, 100, 50, 10
REGS = dict(lam_rate=1e-3, lam_weight=1e-3, lam_dyn=1e-3)

held = group == hg                                         # FingerFreeform frames: never used in training
trn, tst = train & ~held, test & ~held
omean, ocomps = ref02.pca_fit(pose[trn & pose_valid], D_OUT)
Zout = ref02.pca_transform(pose, omean, ocomps); zsd = Zout[trn & pose_valid].std(axis=0)
Y_rnn = Zout / zsd                                         # (N, 6) standardised targets
to_deg = lambda Yh: ref02.pca_inverse(Yh * zsd, omean, ocomps)
L_emg = np.log(emg + 1e-3); U_rnn = (L_emg - L_emg[trn].mean(0)) / L_emg[trn].std(0)   # (N, 16)
T_rnn = np.eye(G)[group]                                   # (N, G) one-hot task input


def train(p: dict, starts: np.ndarray, epochs: int, lr: float, T_: np.ndarray = T_rnn, names: list[str] | None = None,
          length: int = CHUNK, seed: int = SEED) -> list[float]:
    torch.manual_seed(seed); rng_t = np.random.default_rng(seed)
    opt = torch.optim.Adam([p[k] for k in (names or list(p))], lr=lr)
    idx = starts[:, None] + np.arange(length)[None, :]
    Ut, Tt, Yt = (torch.tensor(a[idx], dtype=torch.float32) for a in (U_rnn, T_, Y_rnn))
    Mt = torch.tensor(pose_valid[idx]); Mt[:, :BURN] = False
    hist = []
    for _ in range(epochs):
        order, losses = rng_t.permutation(len(starts)), []
        for i in range(0, len(order), 64):
            b = order[i:i + 64]
            Yh, R = rnn_run(Ut[b], Tt[b], p, ALPHA)
            loss, mse = rnn_loss(Yh, Yt[b], Mt[b], R, p, **REGS)
            opt.zero_grad(); loss.backward(); opt.step(); losses.append(float(mse.detach()))
        hist.append(np.mean(losses))
    return hist


def predict(p: dict, U_: np.ndarray = U_rnn, T_: np.ndarray = T_rnn, recs: np.ndarray | None = None
            ) -> tuple[np.ndarray, np.ndarray]:
    out, rates = np.zeros((len(U_), D_OUT)), np.zeros((len(U_), H))
    with torch.no_grad():
        for r in (np.unique(rec) if recs is None else recs):
            i = np.where(rec == r)[0]
            Yh, R = rnn_run(torch.tensor(U_[i][None], dtype=torch.float32), torch.tensor(T_[i][None], dtype=torch.float32), p, ALPHA)
            out[i], rates[i] = Yh[0].numpy(), R[0].numpy()
    return out, rates


p_rnn = ref.rnn_params(16, G, H, D_OUT, seed=SEED)
hist = train(p_rnn, ref.chunk_starts(rec, trn, CHUNK, STRIDE), EPOCHS, LR)
print("training MSE per pass (standardised units):", np.round(hist, 3))

### 📊 Task 3b · Does the RNN read the EMG? (given)
Scored on the **test** part of every training-task recording (valid pose), per independent block, against:
the global median pose; each recording's own median (the strongest "no EMG" baseline); a static ridge map from lagged
log-EMG **plus the same task input** (does the RNN's memory add anything?); and the trained RNN fed **circularly
shifted** EMG or **no** EMG (does it use the EMG at all, or just the task input?). Differences are RNN minus reference,
mean over blocks with a 95% CI. **What to look for:** a CI entirely below 0 means the RNN is better.

In [ ]:
pred_rnn, rates_rnn = predict(p_rnn)
U_shift = protocol.circular_shift_null(U_rnn, rec, np.random.default_rng(SEED))
pred_shift, _ = predict(p_rnn, U_=U_shift)
pred_zero, _ = predict(p_rnn, U_=np.zeros_like(U_rnn))
X_static = np.concatenate([ref06.lag_features(U_rnn, rec, [0, 5, 10]), T_rnn], axis=1)
Bs, xs_m, ys_m, _ = ref06.rrr_fit(X_static[trn & pose_valid], pose[trn & pose_valid], rank=20)
candidates = {
    "RNN": to_deg(pred_rnn),
    "global median": protocol.global_median_baseline(pose, trn & pose_valid),
    "per-recording median": protocol.per_recording_median_baseline(pose, rec, trn & pose_valid),
    "static ridge + task": ref06.rrr_predict(X_static, Bs, xs_m, ys_m),
    "RNN, shifted EMG": to_deg(pred_shift),
    "RNN, no EMG": to_deg(pred_zero),
}
m_eval = tst & pose_valid
blocks_eval = np.unique(block[m_eval])
per_block = {k: np.array([np.abs(pose[m_eval & (block == b)] - v[m_eval & (block == b)]).mean() for b in blocks_eval])
             for k, v in candidates.items()}
for k, v in per_block.items():
    line = f"{k:22s} {v.mean():6.2f}°"
    if k != "RNN":
        mu, lo, hi = protocol.session_bootstrap_ci(per_block["RNN"] - v)
        line += f"   RNN − this: {mu:+.2f}° [{lo:+.2f}, {hi:+.2f}]"
    print(line)

fig, ax = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
r0 = np.unique(rec[m_eval])[0]; i0 = np.where((rec == r0) & test)[0]
for a, j in zip(ax, [9, 2]):
    a.plot(np.where(pose_valid[i0], pose[i0, j], np.nan), color="k", lw=1, label="true")
    a.plot(candidates["RNN"][i0, j], color="C1", lw=1, label="RNN")
    a.plot(candidates["per-recording median"][i0, j], color="C0", ls="--", lw=1, label="recording median")
    a.set_ylabel(f"joint {j} (°)")
ax[0].legend(fontsize=8); ax[1].set_xlabel("frame (test part of one recording)"); plt.tight_layout(); plt.show()

✍️ **Q3.** Does the RNN beat each recording's own median and the static ridge map that gets the same
task input? Compare "RNN, shifted EMG" and "RNN, no EMG" with "per-recording median": how much of the RNN's accuracy
comes from the EMG, and how much from simply knowing which task is running?

> *Your answer:*

---
### Task 4 · Compositional transfer: learn a new task from its input vector alone
The paper's test of composition: freeze the network and fit **only** the new task's input vector (64 numbers). If the
network has learned reusable pieces, a new task should be reachable by a new *combination* of them, from very little
data. Compare four ways to handle `FingerFreeform`, which the network has never seen, as a function of how many
seconds of it we may use for calibration (`protocol.calibration_splits`: calibrate on the start of each recording's
first half, always score the second half):

| Method | What is fitted on the calibration seconds |
|---|---|
| no task input | nothing (`e_task` = 0) |
| nearest task | nothing; pick the existing task vector that fits the calibration seconds best |
| **task vector only** | the new column of `W_task` (everything else frozen) |
| all weights | every weight, starting from the pretrained network |

Plus a "calibrated median" baseline: the median pose of the calibration seconds. Write `calibrate`, which adapts a
**copy** of the network.

In [ ]:
def calibrate(p: dict, method: str, new_task: int, starts: np.ndarray, epochs: int = 60, lr: float = 1e-2) -> dict:
    """Return an adapted COPY of the pretrained network p (p itself must not change).

    method "vector": copy, set column `new_task` of W_task to 0, train only "W_task" with `train(...)` on `starts`
    method "all":    copy, train every parameter with `train(...)` on `starts`
    Use ref.clone_params(p) for the copy, and pass T_=T_new (the one-hot of the new task) and length=50 to train.
    """
    # TODO
    raise NotImplementedError

#### 💡 Stuck on Task 4? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. Make a copy with `ref.clone_params(p)`.
2. For "vector": zero that column (inside `torch.no_grad()`), and train with `names=["W_task"]`. Only the new column
   gets a gradient, because only it is active when the task input is the new one-hot.
3. For "all": train with `names=None`. Return the copy.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `with torch.no_grad(): q["W_task"][:, k] = 0.0` changes a weight without recording it for gradients.
- `train(q, starts, epochs, lr, T_=T_new, names=[...], length=50)` is the given training loop.

```python
q = ref.clone_params(p_rnn)          # independent copy: changing q never touches p_rnn
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def calibrate(p, method, new_task, starts, epochs=60, lr=1e-2):
    q = ref.clone_params(____)
    if method == "vector":
        with torch.no_grad():
            q["W_task"][:, new_task] = ____
        train(q, starts, epochs, lr, T_=T_new, names=["____"], length=50)
    else:
        train(q, starts, epochs, lr, T_=T_new, names=None, length=50)
    return q
```

</details>

In [ ]:
# Given: the calibration curve (about 40 s)
T_new = T_rnn.copy(); T_new[held] = np.eye(G)[hg]                       # the held-out task gets its own (new) one-hot
T_none = T_rnn.copy(); T_none[held] = 0.0
h_recs = np.unique(rec[held])
SECONDS = [0, 2, 5, 10, 20]
splits = protocol.calibration_splits(held, rec, SECONDS)

def deg_err(P_hat: np.ndarray, mask: np.ndarray) -> float:
    return float(np.abs(pose[mask] - P_hat[mask]).mean())

def run_held(p: dict, T_: np.ndarray) -> np.ndarray:
    out, _ = predict(p, T_=T_, recs=h_recs); return to_deg(out)

pred_none = run_held(p_rnn, T_none)
pred_by_task = {}
for g in range(G):                                                      # every existing task vector, for "nearest task"
    if g != hg:
        Tg = T_rnn.copy(); Tg[held] = np.eye(G)[g]; pred_by_task[g] = run_held(p_rnn, Tg)

curve = {k: [] for k in ["calibrated median", "no task input", "nearest task", "task vector only", "all weights"]}
for s, cal, ev in splits:
    cm, em = cal & pose_valid, ev & pose_valid
    med = np.median(pose[cm], axis=0) if cm.any() else np.median(pose[trn & pose_valid], axis=0)
    curve["calibrated median"].append(float(np.abs(pose[em] - med).mean()))
    curve["no task input"].append(deg_err(pred_none, em))
    if s == 0:
        for k in ["nearest task", "task vector only", "all weights"]:
            curve[k].append(curve["no task input"][-1])
        continue
    best = min(pred_by_task, key=lambda g: deg_err(pred_by_task[g], cm))
    curve["nearest task"].append(deg_err(pred_by_task[best], em))
    st = ref.chunk_starts(rec, cal, 50, 25)
    curve["task vector only"].append(deg_err(run_held(calibrate(p_rnn, "vector", hg, st), T_new), em))
    curve["all weights"].append(deg_err(run_held(calibrate(p_rnn, "all", hg, st), T_new), em))
for k, v in curve.items():
    print(f"{k:18s}", "  ".join(f"{s:>2}s {e:5.2f}" for s, e in zip(SECONDS, v)))

In [ ]:
# ✅ Check Task 4
q_v = calibrate(p_rnn, "vector", hg, ref.chunk_starts(rec, splits[1][1], 50, 25), epochs=2)
assert all(torch.equal(q_v[k], p_rnn[k]) for k in p_rnn if k != "W_task"), "'vector' must leave every other weight unchanged"
changed = (q_v["W_task"] != p_rnn["W_task"]).any(dim=0)
others = torch.tensor(np.arange(G) != hg)
assert changed[hg] and not changed[others].any(), "'vector' must change only the new task's column"
assert torch.equal(p_rnn["W_task"], ref.clone_params(p_rnn)["W_task"]), "calibrate must not modify the pretrained network"
print("✅ Task 4 works: 'vector' changes only the new task's input column")

# 📊 The calibration curve
fig, ax = plt.subplots(figsize=(7, 4))
for k, v in curve.items():
    ax.plot(SECONDS, v, "o-", label=k, lw=2.5 if k == "task vector only" else 1.2)
ax.set_xlabel("seconds of the new task used for calibration"); ax.set_ylabel("error on its other half (°)")
ax.set_title(f"Learning {HELD_OUT}, which the network never saw"); ax.legend(fontsize=8); plt.tight_layout(); plt.show()

✍️ **Q4.** With 2–5 seconds of the new task, which method wins? Where does "all weights" overtake
"task vector only", and why might fitting 64 numbers beat fitting ~6,000 when data are scarce? This is one held-out
task from one person, so there is no confidence interval: what would you need to run before calling it evidence for
H1?

> *Your answer:*

---
### Task 5 · Do tasks share the network's activity space?
Lazzari & Saxena found that tasks share a low-dimensional activity space in the network, and that **muscle** activity
was *less* shared across tasks than the network's. Measure the same three things here, per task group (training
tasks, training frames): the EMG **input**, the RNN **rates**, and the pose **output** (its 6 PCA scores).

For groups a and b, with each group's top-d principal subspace (d = 3):

$$\text{ratio}(a, b) = \frac{\text{variance of } a \text{ captured by } b\text{'s subspace}}{\text{variance of } a
\text{ captured by its own}}$$

`ref.random_ratio` gives the same ratio for random subspaces, and `ref.sharing_index` rescales the mean off-diagonal
ratio so 0 = random and 1 = as good as the group's own.

In [ ]:
def top_subspace(R: np.ndarray, d: int) -> np.ndarray:
    """(D, d) orthonormal basis of the top-d principal components of R (N, D) (centre first)."""
    # TODO
    raise NotImplementedError


def captured_variance(R: np.ndarray, Q: np.ndarray) -> float:
    """Variance of centred R captured by the orthonormal basis Q: sum of squared projections / N."""
    # TODO
    raise NotImplementedError


def cross_ratio(R_by_group: list[np.ndarray], d: int) -> np.ndarray:
    """(G, G) matrix: ratio(a, b) = captured_variance(R_a, Q_b) / captured_variance(R_a, Q_a)."""
    # TODO
    raise NotImplementedError

#### 💡 Stuck on Task 5? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. `top_subspace`: centre R, take the SVD, keep the first d right singular vectors as columns.
2. `captured_variance`: centre R, project on Q, square, sum, divide by the number of rows.
3. `cross_ratio`: compute every group's subspace once, then fill the matrix with the ratio.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `np.linalg.svd(Rc, full_matrices=False)` returns `U, S, Vt`; the top-d directions are `Vt[:d].T`.

```python
R = np.array([[1., 0.], [-1., 0.], [0., 0.1]])
np.linalg.svd(R - R.mean(0), full_matrices=False)[2][:1].T     # ≈ [[±1], [0]]: the x direction
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def top_subspace(R, d):
    _, _, Vt = np.linalg.svd(R - R.mean(axis=0), full_matrices=False)
    return Vt[:____].T

def captured_variance(R, Q):
    Rc = R - R.mean(axis=0)
    return float(((Rc @ ____) ** 2).sum() / len(R))

def cross_ratio(R_by_group, d):
    Qs = [top_subspace(R, d) for R in R_by_group]
    G_ = len(R_by_group)
    out = np.zeros((G_, G_))
    for a in range(G_):
        own = captured_variance(R_by_group[a], Qs[a])
        for b in range(G_):
            out[a, b] = captured_variance(R_by_group[a], Qs[____]) / own
    return out
```

</details>

In [ ]:
# ✅ Check Task 5
Rs_c = [np.random.default_rng(i).normal(size=(40, 6)) * np.r_[3, 2, 1, 1, 1, 1] for i in range(3)]
assert np.allclose(np.abs(top_subspace(Rs_c[0], 2).T @ ref.top_subspace(Rs_c[0], 2)).round(6).max(axis=1), 1), \
    "top_subspace spans a different space than the reference"
assert abs(captured_variance(Rs_c[0], ref.top_subspace(Rs_c[0], 2)) - ref.captured_variance(Rs_c[0], ref.top_subspace(Rs_c[0], 2))) < 1e-9
assert np.allclose(cross_ratio(Rs_c, 2), ref.cross_ratio(Rs_c, 2), atol=1e-8)
assert np.allclose(np.diag(cross_ratio(Rs_c, 2)), 1), "the diagonal must be 1 (own subspace)"
print("✅ Task 5 works")

# Given: sharing of input, network and output across the training task groups
D_SUB = 3
train_groups = [g for g in range(G) if g != hg]
pieces = {"EMG input": U_rnn, "RNN rates": rates_rnn, "pose output": Y_rnn}
sharing, ratios = {}, {}
for name, A in pieces.items():
    by_g = [A[trn & (group == g)] for g in train_groups]
    ratios[name] = cross_ratio(by_g, D_SUB)
    rand = ref.random_ratio(by_g, D_SUB, np.random.default_rng(SEED))
    sharing[name] = ref.sharing_index(ratios[name], rand)
    print(f"{name:12s} sharing index {sharing[name]:.2f}  (mean off-diagonal ratio {ratios[name][~np.eye(len(by_g), dtype=bool)].mean():.2f}, random {rand:.2f})")

fig, ax = plt.subplots(1, 3, figsize=(15, 4.5))
for a, (name, Rm) in zip(ax, ratios.items()):
    im = a.imshow(Rm, vmin=0, vmax=1, cmap="magma"); a.set_title(f"{name}: index {sharing[name]:.2f}")
    a.set_xlabel("subspace of task b"); a.set_ylabel("activity of task a")
plt.colorbar(im, ax=ax, fraction=0.02); plt.show()

✍️ **Q5.** Rank the three (EMG input, RNN rates, pose output) by how much the tasks share them. Lazzari
& Saxena found muscle activity *less* shared than network activity. Do you see the same ordering? Remember notebook 04:
in our data the EMG was *more* shared than the pose. What could make a network's activity look shared even if the
tasks were different (think about what the regularisers push it towards)?

> *Your answer:*

---
## ⚠️ Methodology: what this notebook can and can't tell you
Read this before writing the Result. Each row is a reason the numbers could look better (or different) than the truth, and what would test it.

| Issue | Why it matters | What would test it |
|---|---|---|
| **One person, 14 task groups** | Motif reuse and transfer are within one person; every 'task' is a ~1-minute recording pair | The same analysis on several people (stage 7) |
| **Transfer is one held-out task** | The calibration curve has no CI: a single task can favour either method by luck | Repeat with every task group held out (14 trainings, ~5 min) and bootstrap over blocks |
| **The task one-hot is a strong cue** | Knowing the task already gives the task's typical posture (the per-recording median is a hard baseline), so a good RNN score may say little about the EMG | The 'no EMG' and 'shifted EMG' rows of Task 3b |
| **Hard-EM motifs pick their own data** | Each segment chooses its best motif, so training fit always improves with K; this is why K is chosen on held-out segments, with the motif inferred from the first half only | A proper rSLDS with a persistence prior (e.g. the ssm package) |
| **Small, short-trained RNN** | 64 units and 12 passes were chosen for speed; results may change with size, seed and regulariser weights | Several seeds; a sweep of H and the λs on a validation slice of the training data |
| **Zero-phase envelope** | The EMG envelope was filtered forwards and backwards, so each frame holds ~100 ms of future EMG; a 'causal' RNN still sees a little of the future | Rebuild the envelope with a causal filter for any online claim |
| **Low-dimensional state** | Motifs live in 4 pose PCs (notebook 02: they lose ~9°), and the input is notebook 06's 3 shared latents | Repeat with more PCs, or in the shared-subspace pose coordinates |

---
## Result · gate for stage 5
**Question:** Do tasks reuse dynamical building blocks, and can a new task be learned by recombining them?

| Measure | Value (mean, 95% CI over sessions) | Baseline / null | Pass? |
|---|---|---|---|
| Held-out one-step error: best K vs K = 1 | | | |
| Motif reuse: groups per motif; mutual information vs null | | | |
| Held-out task: own ÷ other-task library | | | |
| RNN − per-recording median (CI over blocks) | | | |
| RNN − RNN with shifted EMG (CI) | | | |
| Calibration at 5 s: task vector only vs no task / all weights | | | |
| Sharing index: EMG input / RNN rates / pose output | | | |

**Gate:** (a) some K > 1 beats K = 1 on held-out segments; (b) the RNN beats the per-recording median and its shifted-EMG input (both CIs below 0); (c) the task vector beats no task input with ≤ 5 s of a new task.

- **Pass →** stage 6 (notebook 08): use the best stage-5 decoder for the bottleneck comparison, and repeat Task 4 for every held-out task with bottlenecked inputs.
- **Fail →** report which part failed. Motifs that describe but don't predict, or an RNN that only uses the task cue, are findings: they say the shared structure in this data is descriptive, not a better decoder.
- **Was your prediction right? Why or why not?**